In [1]:
# Verificar qué hay en la carpeta models
import os
print(os.listdir("../models"))

['.gitignore', '.ipynb_checkpoints', '09_models.ipynb', 'baseline_decision_tree.pkl', 'baseline_knn.pkl', 'baseline_logistic_regression.pkl', 'baseline_random_forest.pkl', 'baseline_svm.pkl', 'preprocessing_pipeline.pkl', 'preprocessing_pipeline.pkl.dvc']


In [1]:
import joblib
import os

print("🔍 Cargando modelo...")
modelo = joblib.load("../models/baseline_logistic_regression.pkl")
print("✅ ÉXITO: Modelo cargado")
print(f"   Tipo: {type(modelo)}")

🔍 Cargando modelo...
✅ ÉXITO: Modelo cargado
   Tipo: <class 'imblearn.pipeline.Pipeline'>


C:\Users\winx_\anaconda3\envs\proyecto\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\winx_\anaconda3\envs\proyecto\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\winx_\anaconda3\envs\proyecto\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.7.2 when us

In [4]:
# ============================================
# PB-11 - VERSIÓN CORREGIDA
# ============================================

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas")

# ============================================
# 1. CARGAR DATOS RAW
# ============================================
print("\n📂 Cargando datos raw...")
df_raw = pd.read_csv("../data/raw/dataset_pucp.csv", nrows=10000)
print(f"✅ Datos raw: {df_raw.shape}")

# Identificar target
target = 'Revenue' if 'Revenue' in df_raw.columns else df_raw.columns[-1]
print(f"🎯 Target: {target}")

y = df_raw[target]
X_raw = df_raw.drop(columns=[target])
print(f"📊 X_raw shape: {X_raw.shape}")

# ============================================
# 2. INTENTAR USAR EL PIPELINE
# ============================================
print("\n🔍 Intentando cargar pipeline...")

try:
    pipeline = joblib.load("../models/preprocessing_pipeline.pkl")
    print(f"✅ Pipeline cargado: {type(pipeline)}")
    
    # Intentar transformar
    print("🔄 Aplicando pipeline...")
    X_processed = pipeline.transform(X_raw)
    print(f"✅ X_processed shape: {X_processed.shape}")
    usar_pipeline = True
    
except Exception as e:
    print(f"⚠️ No se pudo usar el pipeline: {e}")
    print("   Usando preprocesamiento manual...")
    usar_pipeline = False

# ============================================
# 3. SI EL PIPELINE NO FUNCIONA, PREPROCESAMIENTO MANUAL
# ============================================
if not usar_pipeline:
    print("\n🛠️ Preprocesamiento manual...")
    
    # Solo columnas numéricas
    X = X_raw.select_dtypes(include=[np.number])
    
    # Rellenar nulos
    for col in X.columns:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())
    
    # Escalar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ X_scaled shape: {X_scaled.shape}")

# ============================================
# 4. CARGAR MODELOS DE LA COMPAÑERA (opcional)
# ============================================
print("\n🔍 Intentando cargar modelos baseline...")

modelos_externos = {}
try:
    modelos_externos['Logistic Regression'] = joblib.load("../models/baseline_logistic_regression.pkl")
    modelos_externos['Random Forest'] = joblib.load("../models/baseline_random_forest.pkl")
    modelos_externos['Decision Tree'] = joblib.load("../models/baseline_decision_tree.pkl")
    modelos_externos['SVM'] = joblib.load("../models/baseline_svm.pkl")
    modelos_externos['KNN'] = joblib.load("../models/baseline_knn.pkl")
    print(f"✅ {len(modelos_externos)} modelos cargados")
    usar_modelos_externos = True
except Exception as e:
    print(f"⚠️ No se pudieron cargar modelos externos: {e}")
    usar_modelos_externos = False

# ============================================
# 5. EVALUAR MODELOS
# ============================================
resultados = []

if usar_pipeline and usar_modelos_externos:
    # Usar pipeline + modelos externos
    print("\n📊 Evaluando con pipeline + modelos externos...")
    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    for nombre, modelo in modelos_externos.items():
        try:
            y_pred = modelo.predict(X_test)
            resultados.append({
                'Modelo': nombre,
                'Accuracy': round(accuracy_score(y_test, y_pred), 4),
                'F1_Score': round(f1_score(y_test, y_pred), 4)
            })
            print(f"{nombre}: F1={resultados[-1]['F1_Score']:.4f}")
        except Exception as e:
            print(f"❌ {nombre}: Error - {e}")

else:
    # Usar modelos propios (entrenados ahora)
    print("\n📊 Entrenando y evaluando modelos propios...")
    
    # Preparar datos
    if usar_pipeline:
        X_data = X_processed
    else:
        X_data = X_scaled
    
    X_train, X_test, y_train, y_test = train_test_split(X_data, y, test_size=0.2, random_state=42)
    
    # Modelos propios
    modelos_propios = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Decision Tree': DecisionTreeClassifier(),
        'Random Forest': RandomForestClassifier(),
        'SVM': SVC(),
        'KNN': KNeighborsClassifier()
    }
    
    for nombre, modelo in modelos_propios.items():
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        resultados.append({
            'Modelo': nombre,
            'Accuracy': round(accuracy_score(y_test, y_pred), 4),
            'F1_Score': round(f1_score(y_test, y_pred), 4)
        })
        print(f"{nombre}: F1={resultados[-1]['F1_Score']:.4f}")

# ============================================
# 6. TABLA DE RESULTADOS
# ============================================
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('F1_Score', ascending=False)

print("\n" + "="*60)
print("📈 TABLA COMPARATIVA (ordenada por F1-Score)")
print("="*60)
print(df_resultados.to_string(index=False))

# ============================================
# 7. GUARDAR RESULTADOS
# ============================================
import os
os.makedirs("../reports", exist_ok=True)
df_resultados.to_csv("../reports/model_evaluation_results.csv", index=False)

print("\n✅ Resultados guardados en: ../reports/model_evaluation_results.csv")
print("✅ PB-11 COMPLETADO")

✅ Librerías cargadas

📂 Cargando datos raw...
✅ Datos raw: (10000, 18)
🎯 Target: Revenue
📊 X_raw shape: (10000, 17)

🔍 Intentando cargar pipeline...
✅ Pipeline cargado: <class 'imblearn.pipeline.Pipeline'>
🔄 Aplicando pipeline...
⚠️ No se pudo usar el pipeline: This 'Pipeline' has no attribute 'transform'
   Usando preprocesamiento manual...

🛠️ Preprocesamiento manual...
✅ X_scaled shape: (10000, 14)

🔍 Intentando cargar modelos baseline...
✅ 5 modelos cargados

📊 Entrenando y evaluando modelos propios...
Logistic Regression: F1=0.5287
Decision Tree: F1=0.5394
Random Forest: F1=0.6536
SVM: F1=0.6034
KNN: F1=0.5670

📈 TABLA COMPARATIVA (ordenada por F1-Score)
             Modelo  Accuracy  F1_Score
      Random Forest    0.9030    0.6536
                SVM    0.8955    0.6034
                KNN    0.8885    0.5670
      Decision Tree    0.8540    0.5394
Logistic Regression    0.8850    0.5287

✅ Resultados guardados en: ../reports/model_evaluation_results.csv
✅ PB-11 COMPLETADO


Mejor modelo: Random Forest (F1-Score: 0.6536)
Accuracy alta: Todos los modelos superan 85%
El dataset está desbalanceado (F1 menor que Accuracy)